In [ ]:
import json
import os
import sys
import time
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, '..')
from utils.api_client import create_openrouter_client

In [6]:
#load dotenv
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
INPUT_FILE = '../data/seed_extended.jsonl'
OUTPUT_FILE = '../data/seed_latex_cleaned.jsonl'
MODEL = "openai/gpt-4o"
MAX_WORKERS = 20

client = create_openrouter_client()

SYSTEM_PROMPT = (
    "You are a LaTeX formatting expert. Your task is to rewrite the provided text "
    "EXACTLY as it is, without changing any words, meanings, or punctuation. "
    "The ONLY changes you are permitted to make are:\n"
    "1. Convert mathematical symbols (like \u03b7, \u03b1, \u03b8, etc.) into their proper LaTeX commands (e.g., \\eta, \\alpha, \\theta).\n"
    "2. Ensure all mathematical expressions and symbols are correctly wrapped in $ for inline math or $$ for block math.\n"
    "3. Fix any broken LaTeX delimiters (e.g., matching opening/closing $$).\n"
    "Return only the corrected text."
)

In [11]:
def fix_latex(text):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text}
            ],
            temperature=0,
        )
        #print response status and content for debugging
        print(f"Response received: {response}")
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error processing text: {e}")
        return text

def process_line(line_tuple):
    index, line = line_tuple
    try:
        data = json.loads(line)
        original_question = data.get("question", "")
        print(f"Processing question {index}...")
        
        # Update the question field
        data["question"] = fix_latex(original_question)
        return data
    except json.JSONDecodeError:
        print(f"Skipping malformed line at index {index}")
        return None

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    with open(INPUT_FILE, 'r', encoding='utf-8') as infile:
        lines = infile.readlines()

    # Use ThreadPoolExecutor to process lines in parallel
    indexed_lines = list(enumerate(lines, 1))
    results = []
    
    print(f"Starting parallel processing with {MAX_WORKERS} workers...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # map ensures the results stay in the original order
        results = list(executor.map(process_line, indexed_lines))

    # Write results to file (filtering out any None results from errors)
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as outfile:
        for entry in results:
            if entry:
                outfile.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"\nProcessing complete! Saved to {OUTPUT_FILE}")

main()

Starting parallel processing with 20 workers...
Processing question 1...
Processing question 2...
Processing question 3...
Processing question 4...
Processing question 5...
Processing question 6...
Processing question 7...
Processing question 8...
Processing question 9...
Processing question 10...
Processing question 11...
Processing question 12...
Processing question 13...
Processing question 14...
Processing question 15...
Processing question 16...
Processing question 17...
Processing question 18...
Processing question 19...
Processing question 20...
Response received: ChatCompletion(id='gen-1768512343-44DoA7zitbKLKbncgE2K', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Find the characteristic life necessary for 10\\% failures by 168 h, given a shape parameter of 2.0.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1768512